# 05 · Simulación y búsqueda completa

Tópico 05 del temario: **matrices, subconjuntos y permutaciones**.

## Simulación

**Idea:** el enunciado describe un proceso paso a paso. Lo más seguro es **hacer exactamente lo que dice**, un paso a la vez, con variables que guardan el estado.

Receta:
1. Decide qué **estado** hay que guardar (posición, contador, tablero...).
2. Escribe un bucle que aplique **una regla** por vuelta.
3. Define cuándo **termina**.
4. Comprueba cuántos pasos habrá en el peor caso (¿alcanza el tiempo?).

### Ejemplo: un robot en una cuadrícula

El robot empieza en `(0, 0)` y recibe órdenes `U`, `D`, `L`, `R`. ¿Dónde termina?

In [ ]:
def robot(ordenes):
    x = y = 0
    mov = {"U": (0, 1), "D": (0, -1), "L": (-1, 0), "R": (1, 0)}
    for o in ordenes:
        dx, dy = mov[o]
        x += dx
        y += dy
    return x, y


assert robot("UURDL") == (0, 1)
assert robot("") == (0, 0)

### Ejemplo: el problema de Josefo

`n` personas en círculo; se elimina cada `k`-ésima. ¿Quién sobrevive? Una **cola circular** (`deque`) lo simula.

In [ ]:
from collections import deque


def josefo(n, k):
    ronda = deque(range(1, n + 1))
    while len(ronda) > 1:
        ronda.rotate(-(k - 1))      # los k-1 primeros pasan al final
        ronda.popleft()             # se elimina el k-ésimo
    return ronda[0]


assert josefo(7, 3) == 4
assert josefo(1, 5) == 1

### Ejemplo: vida en una matriz (un paso del "juego de la vida")

Una celda viva (`1`) sigue viva con 2 o 3 vecinos vivos; una muerta (`0`) nace con exactamente 3. **Calcula el nuevo tablero a partir del anterior**, sin modificarlo mientras lo lees.

In [ ]:
DIR8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


def paso_vida(t):
    f, c = len(t), len(t[0])
    nuevo = [[0] * c for _ in range(f)]
    for i in range(f):
        for j in range(c):
            vivos = 0
            for di, dj in DIR8:
                ni, nj = i + di, j + dj
                if 0 <= ni < f and 0 <= nj < c:
                    vivos += t[ni][nj]
            if t[i][j] == 1:
                nuevo[i][j] = 1 if vivos in (2, 3) else 0
            else:
                nuevo[i][j] = 1 if vivos == 3 else 0
    return nuevo


bloque = [[0, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]]
assert paso_vida(bloque) == bloque               # el bloque 2x2 es estable
parpadeo = [[0, 1, 0], [0, 1, 0], [0, 1, 0]]
assert paso_vida(parpadeo) == [[0, 0, 0], [1, 1, 1], [0, 0, 0]]

### Ejemplo: sopa de letras

¿Aparece la palabra en la matriz, en alguna de las 8 direcciones?

In [ ]:
def sopa_de_letras(m, palabra):
    f, c = len(m), len(m[0])
    n = len(palabra)
    for i in range(f):
        for j in range(c):
            for di, dj in DIR8:
                ok = True
                for t in range(n):
                    ni, nj = i + di * t, j + dj * t
                    if not (0 <= ni < f and 0 <= nj < c) or m[ni][nj] != palabra[t]:
                        ok = False
                        break
                if ok:
                    return True
    return False


grid = ["abc", "def", "ghi"]
assert sopa_de_letras(grid, "ceg")          # diagonal
assert sopa_de_letras(grid, "fed")          # derecha a izquierda
assert not sopa_de_letras(grid, "xyz")

## Búsqueda completa (fuerza bruta)

**Idea:** si no sabes cómo hacerlo de forma inteligente, **prueba todas las posibilidades** y quédate con la mejor o la que cumple. Funciona cuando el número de posibilidades es pequeño.

Antes de escribirla, **cuenta las posibilidades**:

| Qué pruebas | Cuántas posibilidades |
|---|---|
| todos los pares | `n²` |
| todas las ternas | `n³` |
| todos los subconjuntos | `2^n` (llega hasta `n ≈ 20`) |
| todas las permutaciones | `n!` (llega hasta `n ≈ 9 o 10`) |
| todas las asignaciones de `k` opciones a `n` lugares | `k^n` |

### Probar todos los pares

In [ ]:
def pares_con_suma(arr, k):
    cuenta = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):        # j > i: cada par una sola vez
            if arr[i] + arr[j] == k:
                cuenta += 1
    return cuenta


assert pares_con_suma([1, 2, 3, 4, 5], 6) == 2          # (1,5) (2,4)

## Subconjuntos

Cada elemento está **dentro** o **fuera**: `2^n` subconjuntos.

### Con máscaras de bits

Cada número de `0` a `2^n - 1` representa un subconjunto: el bit `i` indica si el elemento `i` está dentro.

In [ ]:
def subconjuntos(arr):
    n = len(arr)
    res = []
    for mascara in range(1 << n):
        sub = [arr[i] for i in range(n) if mascara >> i & 1]
        res.append(sub)
    return res


assert subconjuntos([1, 2]) == [[], [1], [2], [1, 2]]
assert len(subconjuntos(list(range(10)))) == 1024

### Problema: ¿algún subconjunto suma `k`?

In [ ]:
def existe_subconjunto_suma(arr, k):
    n = len(arr)
    for mascara in range(1 << n):
        s = 0
        for i in range(n):
            if mascara >> i & 1:
                s += arr[i]
        if s == k:
            return True
    return False


assert existe_subconjunto_suma([3, 34, 4, 12, 5, 2], 9)       # 4 + 5
assert not existe_subconjunto_suma([3, 34, 4], 100)

### Con `itertools`

In [ ]:
from itertools import combinations, permutations, product

# Subconjuntos de tamaño exacto k
assert list(combinations([1, 2, 3], 2)) == [(1, 2), (1, 3), (2, 3)]

# Todos los subconjuntos por tamaño
def todos_los_subconjuntos(arr):
    res = []
    for k in range(len(arr) + 1):
        res.extend(combinations(arr, k))
    return res


assert len(todos_los_subconjuntos([1, 2, 3])) == 8

# Producto cartesiano: todas las asignaciones
assert list(product("ab", repeat=2)) == [("a", "a"), ("a", "b"), ("b", "a"), ("b", "b")]

### Con recursión (decidir elemento por elemento)

Para cada elemento: lo **tomo** o **no lo tomo**. Útil porque permite **podar** (dejar de explorar cuando ya no puede funcionar).

In [ ]:
def cuenta_subconjuntos_suma(arr, k):
    def rec(i, suma):
        if i == len(arr):
            return 1 if suma == k else 0
        return rec(i + 1, suma) + rec(i + 1, suma + arr[i])      # no tomo / tomo
    return rec(0, 0)


assert cuenta_subconjuntos_suma([1, 2, 3, 4], 5) == 2           # {1,4} {2,3}

## Permutaciones

Todas las formas de ordenar `n` elementos: `n!`.

In [ ]:
assert list(permutations([1, 2, 3])) == [
    (1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)
]


# Sin librería: recursión con "usado"
def permutaciones(n):
    res = []
    actual = []
    usado = [False] * n

    def rec():
        if len(actual) == n:
            res.append(actual[:])
            return
        for x in range(n):
            if not usado[x]:
                usado[x] = True
                actual.append(x)
                rec()
                actual.pop()                 # deshacer ("backtracking")
                usado[x] = False

    rec()
    return res


assert permutaciones(3) == [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]

### Problema: ¿cuántas permutaciones cumplen una regla?

Cuenta las permutaciones de `1..n` donde ningún elemento queda en su posición original (*desarreglos*).

In [ ]:
def contar_desarreglos(n):
    cuenta = 0
    for p in permutations(range(1, n + 1)):
        if all(p[i] != i + 1 for i in range(n)):
            cuenta += 1
    return cuenta


assert [contar_desarreglos(n) for n in range(1, 7)] == [0, 1, 2, 9, 44, 265]

### Problema: mejor orden de tareas

El orden importa: probar **todos** los órdenes (con `n ≤ 8`) y quedarse con el mejor.

In [ ]:
def menor_costo_ruta(dist):
    """dist[i][j] = costo de ir de i a j. Visita todas las ciudades empezando en 0, sin volver."""
    n = len(dist)
    mejor = float("inf")
    for p in permutations(range(1, n)):
        costo = 0
        actual = 0
        for ciudad in p:
            costo += dist[actual][ciudad]
            actual = ciudad
        mejor = min(mejor, costo)
    return mejor


d = [[0, 2, 9, 10], [1, 0, 6, 4], [15, 7, 0, 8], [6, 3, 12, 0]]
assert menor_costo_ruta(d) == 16
print("ok")

## Poda: no explorar lo que ya no sirve

Si en un punto ya sabes que no puedes cumplir, **corta** esa rama.

In [ ]:
def subconjunto_suma_con_poda(arr, k):
    """arr con positivos. Si la suma ya pasó de k, no sigas."""
    def rec(i, suma):
        if suma == k:
            return True
        if suma > k or i == len(arr):
            return False                           # poda
        return rec(i + 1, suma + arr[i]) or rec(i + 1, suma)
    return rec(0, 0)


assert subconjunto_suma_con_poda([3, 34, 4, 12, 5, 2], 9)
assert not subconjunto_suma_con_poda([3, 34, 4], 100)

## ¿Simulación o búsqueda completa?

| Si el enunciado... | Piensa en |
|---|---|
| describe un proceso paso a paso | **simulación**: haz lo que dice |
| pide "el mejor/cualquier/cuántos" y `N` es pequeño | **búsqueda completa** |
| `N ≤ 20` y elegir elementos | **subconjuntos** (máscaras o recursión) |
| `N ≤ 9` y importa el orden | **permutaciones** |
| una matriz con reglas locales | simulación con **direcciones** |

## Para practicar

1. `triangulos(arr)`: cuántas ternas de longitudes pueden formar un triángulo (`a + b > c`).
2. `particion_igual(arr)`: ¿se puede dividir en dos grupos de igual suma? (subconjuntos).
3. `n_reinas(n)`: cuántas formas de colocar `n` reinas sin atacarse, `n ≤ 8` (backtracking).
4. `caminos_en_cuadricula(f, c)`: cuántos caminos de `(0,0)` a `(f-1,c-1)` moviéndote solo a la derecha o abajo (recursión; después piensa cómo mejorarla).
5. `siguiente_estado(tablero)`: cualquier simulación que te pongan, siguiendo el enunciado al pie de la letra.

## Errores típicos

- Calcular la cantidad de posibilidades **después** de programar: hazlo **antes**.
- Modificar el tablero mientras lo lees en una simulación (crea uno nuevo).
- Pares duplicados: usa `j > i`.
- Olvidar deshacer en recursión (`actual.pop()`, `usado[x] = False`).
- Recursión demasiado profunda: ver `sys.setrecursionlimit`.